# Experiment 8: Clustering Human Activity Recognition Data using K-Means, DBSCAN and Hierarchical Clustering

```
experiment8_har_clustering.py
=============================
ICS1512 - Machine Learning Algorithms Laboratory
Experiment 8 (Lab Manual title: "Clustering Human Activity Recognition
Data using K-Means, DBSCAN, and Hierarchical Clustering")

Model A : K-Means (k chosen by the Elbow method + silhouette analysis)
Model B : DBSCAN (eps / min_samples tuned on a PCA-reduced space)
Model C : Hierarchical Agglomerative Clustering (Ward linkage, dendrogram)

Dataset : UCI Human Activity Recognition Using Smartphones (10,299 windows,
          561 engineered time/frequency-domain features, 6 activities).
Uses the reusable module ml_lab_utils.py (from Experiment 1) for:
    - the mandatory plot style    -> set_plot_style(), _bold_axis_labels()
    - 600 DPI EPS figure export   -> _save_eps()
```

In [ ]:
import json
import os
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (adjusted_rand_score, calinski_harabasz_score,
                             davies_bouldin_score, normalized_mutual_info_score,
                             silhouette_score)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
FIG_DIR = "figures"
RES_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

## 1. LOAD DATASET (extract the archive on first run)

In [ ]:
DATA_ZIP = "har_dataset.zip"
DATA_CSV = "har_data.csv"

if not os.path.exists(DATA_CSV) and os.path.exists(DATA_ZIP):
    with zipfile.ZipFile(DATA_ZIP) as zf:
        zf.extractall(".")
    print(f"Extracted {DATA_ZIP}")

df = pd.read_csv(DATA_CSV)
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print(df["Activity"].value_counts())

feature_cols = [c for c in df.columns if c != "Activity"]
X_raw = df[feature_cols].values
activities = np.array(sorted(df["Activity"].unique()))
activity_to_idx = {a: i for i, a in enumerate(activities)}
y_true = df["Activity"].map(activity_to_idx).values
print("Feature range: [%.3f, %.3f]" % (X_raw.min(), X_raw.max()))

## 2. PREPROCESSING

The published features are already bounded to [-1, 1], but their spreads
differ, so every feature is standardized to zero mean / unit variance
before any distance-based algorithm sees it. PCA projections used for
DBSCAN and for the 2-D plots are fit on the standardized matrix.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
print("After standardization: mean = %.2e, std = %.3f" % (X.mean(), X.std()))

pca_full = PCA(random_state=RANDOM_STATE).fit(X)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = int(np.searchsorted(cum_var, 0.95) + 1)
print("Components for 95%% variance: %d of %d" % (n_95, X.shape[1]))
print("PC1 explains %.1f%%, PC1-PC2 explain %.1f%%"
      % (100 * cum_var[0], 100 * cum_var[1]))

X_pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
X_pca10 = PCA(n_components=10, random_state=RANDOM_STATE).fit_transform(X)

# t-SNE is O(n^2)-ish, so it is fit on a stratified subsample for the plots.
rng = np.random.RandomState(RANDOM_STATE)
sub_idx = np.concatenate([rng.choice(np.where(y_true == c)[0], 500, replace=False)
                          for c in range(len(activities))])
rng.shuffle(sub_idx)
t0 = time.time()
X_tsne = TSNE(n_components=2, perplexity=30, init="pca",
              random_state=RANDOM_STATE).fit_transform(X[sub_idx])
print("t-SNE on %d samples: %.0fs" % (len(sub_idx), time.time() - t0))

## 3. EXPLORATORY DATA ANALYSIS

## Reusable utilities (`ml_lab_utils`, from Experiment 1)

Inlined here so this notebook runs on its own without a separate `ml_lab_utils.py`.

In [ ]:
"""
ml_lab_utils.py
================
ICS1512 - Machine Learning Algorithms Laboratory
Reusable utility module used across ALL experiments.

Implements (per lab manual, Section 4):
    1. One reusable EDA function            -> generate_eda_summary()
    2. One reusable Regression function      -> train_evaluate_regression()
    3. One reusable Classification function  -> train_evaluate_classification()
    4. One reusable Regression metrics fn    -> regression_performance_metrics()
    5. One reusable Classification metrics   -> classification_performance_metrics()

Formatting rules enforced everywhere (per lab manual, Section 1):
    - Times New Roman, 15 pt for all text / legends
    - Bold, Times New Roman, 15 pt axis labels
    - Figures exported as .eps at 600 DPI (Section 3)

NOTE on fonts: "Times New Roman" itself is a proprietary Microsoft font and is
not installable on Linux. Liberation Serif is metrically-compatible (identical
glyph widths/kerning) and is registered here under the family name
"Times New Roman" so that rcParams['font.family'] = 'Times New Roman' works
transparently. On Windows/macOS, if the real Times New Roman is installed,
matplotlib will simply use that instead.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------
# 1. GLOBAL PLOT STYLE  (Section 1 of the manual)
# --------------------------------------------------------------------------
def set_plot_style(font_size=15):
    """
    Applies the mandatory lab formatting to every matplotlib figure:
        - Times New Roman (or metric-compatible Liberation Serif) font
        - 15 pt base font size
        - 15 pt Times New Roman legends
        - Bold, 15 pt, Times New Roman axis labels
    Call this once at the start of a notebook / script.
    """
    # Register Liberation Serif under the alias "Times New Roman" if the
    # genuine font is not present on this machine.
    installed_fonts = {f.name for f in fm.fontManager.ttflist}
    if "Times New Roman" not in installed_fonts:
        liberation_paths = [
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-Italic.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSerif-BoldItalic.ttf",
        ]
        for p in liberation_paths:
            if os.path.exists(p):
                fm.fontManager.addfont(p)
                # Force the registered family name to "Times New Roman"
                # (FontEntry is a frozen dataclass in modern matplotlib, so we
                # replace the last-added entry rather than mutate it in place)
                last = fm.fontManager.ttflist[-1]
                fm.fontManager.ttflist[-1] = fm.FontEntry(
                    fname=last.fname, name="Times New Roman",
                    style=last.style, variant=last.variant,
                    weight=last.weight, stretch=last.stretch, size=last.size,
                )

    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "axes.labelsize": font_size,
        "axes.labelweight": "bold",
        "axes.titlesize": font_size,
        "axes.titleweight": "bold",
        "xtick.labelsize": font_size - 2,
        "ytick.labelsize": font_size - 2,
        "figure.titlesize": font_size + 2,
        "savefig.dpi": 600,
        "figure.dpi": 150,   # screen preview; export always forced to 600 (see save)
        "svg.fonttype": "none",
    })


def _bold_axis_labels(ax, xlabel=None, ylabel=None, title=None, fs=15):
    """Helper: apply Times New Roman / Bold / 15pt to a single axis explicitly."""
    fp_bold = fm.FontProperties(family="Times New Roman", weight="bold", size=fs)
    fp_reg = fm.FontProperties(family="Times New Roman", size=fs - 2)
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontproperties=fp_bold)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontproperties=fp_bold)
    if title is not None:
        ax.set_title(title, fontproperties=fp_bold, fontsize=fs)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(fp_reg)
    leg = ax.get_legend()
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontproperties(fp_reg)


def _save_eps(fig, save_path, also_png=True):
    """Export a figure as .eps at 600 DPI (Section 3 of the manual).

    If also_png is True, an additional .png copy is saved alongside the .eps
    (same basename) purely so the figure can be embedded when compiling the
    LaTeX report with pdflatex/xelatex, which cannot rasterize .eps directly
    without Ghostscript. The .eps remains the official, mandated deliverable.
    """
    if save_path is None:
        return None
    if not save_path.lower().endswith(".eps"):
        save_path = os.path.splitext(save_path)[0] + ".eps"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, format="eps", dpi=600, bbox_inches="tight")
    if also_png:
        png_path = os.path.splitext(save_path)[0] + ".png"
        fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
    return save_path


# --------------------------------------------------------------------------
# 2. GENERIC EDA FUNCTION  (Section 4.1)  -> ONE consolidated 12-subplot figure
# --------------------------------------------------------------------------
def generate_eda_summary(df, target_col=None, dataset_name="Dataset",
                          save_path=None, figsize=(22, 16)):
    """
    Generic, reusable EDA function that works on ANY tabular dataset
    (classification, regression, or unlabeled). Produces ONE consolidated
    figure containing 12 EDA subplots on a single page, per Section 2 of the
    lab manual.

    Parameters
    ----------
    df : pandas.DataFrame
        The full dataset (features + target, if any).
    target_col : str or None
        Name of the target/label column, if present. If None, the function
        treats the dataset as unlabeled and adapts the 12-panel layout
        accordingly (no class-distribution / target-correlation panels).
    dataset_name : str
        Used in the figure's suptitle.
    save_path : str or None
        If given, the figure is exported as .eps @ 600 DPI to this path.
    figsize : tuple
        Overall figure size in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    set_plot_style()
    df = df.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)

    is_classification_target = (
        target_col is not None and
        (df[target_col].dtype == "object" or df[target_col].nunique() <= 20)
    )

    # Pick the most "informative" numeric feature (highest variance) as the
    # representative single feature for panels 6/8/9/10, instead of blindly
    # using the first column (which can be degenerate/constant, e.g. corner
    # pixels in an image dataset such as MNIST/Digits).
    if numeric_cols:
        # Prefer genuinely continuous columns (more than 5 distinct values) so
        # binary/near-constant encoded columns (e.g. a 0/1 "sex" flag, or
        # constant corner pixels in image data) are not picked as the
        # representative single feature for panels 6/8/9/10.
        continuous_cols = [c for c in numeric_cols if df[c].nunique() > 5]
        candidate_cols = continuous_cols if continuous_cols else numeric_cols
        variances = df[candidate_cols].var().sort_values(ascending=False)
        top_var_cols = variances.index.tolist()
        feat_a = top_var_cols[0]
        feat_b = top_var_cols[1] if len(top_var_cols) > 1 else top_var_cols[0]
        kde_cols = top_var_cols[:4]
    else:
        feat_a = feat_b = None
        kde_cols = []

    fig = plt.figure(figsize=figsize)
    fig.suptitle(f"Exploratory Data Analysis Summary \u2013 {dataset_name}",
                 fontweight="bold", fontsize=17,
                 fontproperties=fm.FontProperties(family="Times New Roman",
                                                   weight="bold", size=17))
    gs = fig.add_gridspec(3, 4, hspace=0.55, wspace=0.4)
    axes = [fig.add_subplot(gs[i // 4, i % 4]) for i in range(12)]
    panel = 0

    # ---- Panel 1: Dataset overview (head / shape as a text table) ----
    ax = axes[panel]; panel += 1
    ax.axis("off")
    overview_txt = (
        f"Shape: {df.shape[0]} rows x {df.shape[1]} cols\n"
        f"Numeric features: {len(numeric_cols)}\n"
        f"Categorical features: {len(categorical_cols)}\n"
        f"Missing cells: {int(df.isnull().sum().sum())}\n"
        f"Duplicate rows: {int(df.duplicated().sum())}"
    )
    ax.text(0.02, 0.9, overview_txt, va="top", ha="left",
            fontproperties=fm.FontProperties(family="Times New Roman", size=13),
            transform=ax.transAxes)
    _bold_axis_labels(ax, title="1. Dataset Overview")

    # ---- Panel 2: Statistical summary heat-table (mean/std/min/max) ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[["mean", "std", "min", "max"]]
        desc_norm = (desc - desc.min()) / (desc.max() - desc.min() + 1e-9)
        sns.heatmap(desc_norm.iloc[:8], annot=desc.iloc[:8].round(1), fmt="",
                    cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontsize": 8, "fontfamily": "Times New Roman"})
    _bold_axis_labels(ax, title="2. Statistical Summary")

    # ---- Panel 3: Missing value analysis ----
    ax = axes[panel]; panel += 1
    miss = df.isnull().mean().sort_values(ascending=False) * 100
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "No Missing Values", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=14))
        ax.axis("off")
    else:
        miss[miss > 0].head(10).plot(kind="bar", ax=ax, color="#c0392b")
    _bold_axis_labels(ax, "Feature", "% Missing", "3. Missing Value Analysis")

    # ---- Panel 4: Class distribution / target distribution ----
    ax = axes[panel]; panel += 1
    if target_col is not None:
        if is_classification_target:
            df[target_col].value_counts().plot(kind="bar", ax=ax, color="#2980b9")
            _bold_axis_labels(ax, "Class", "Count", "4. Class Distribution")
        else:
            sns.histplot(df[target_col], kde=True, ax=ax, color="#2980b9")
            _bold_axis_labels(ax, target_col, "Frequency", "4. Target Distribution")
    else:
        ax.axis("off")
        ax.text(0.5, 0.5, "No target column supplied", ha="center", va="center",
                fontproperties=fm.FontProperties(family="Times New Roman", size=12))
        _bold_axis_labels(ax, title="4. Target Distribution")

    # ---- Panel 5: Correlation matrix (heatmap) ----
    ax = axes[panel]; panel += 1
    corr_cols = numeric_cols[:10] if len(numeric_cols) > 10 else numeric_cols
    if len(corr_cols) >= 2:
        sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", center=0, ax=ax,
                    cbar=False, annot=len(corr_cols) <= 6, fmt=".2f",
                    annot_kws={"fontsize": 7})
    _bold_axis_labels(ax, title="5. Correlation Matrix")

    # ---- Panel 6: Feature distribution (histogram of 1st numeric feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.histplot(df[feat_a], kde=True, ax=ax, color="#27ae60")
    _bold_axis_labels(ax, feat_a if feat_a else "", "Frequency",
                       "6. Feature Distribution")

    # ---- Panel 7: Box plot (outlier detection) across numeric features ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        plot_cols = numeric_cols[:6]
        df_scaled = (df[plot_cols] - df[plot_cols].mean()) / (df[plot_cols].std() + 1e-9)
        sns.boxplot(data=df_scaled, ax=ax, color="#f39c12")
        ax.tick_params(axis="x", rotation=45)
    _bold_axis_labels(ax, "Feature", "Standardized Value", "7. Box Plot (Outliers)")

    # ---- Panel 8: Violin plot ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        sns.violinplot(y=df[feat_a], ax=ax, color="#8e44ad")
    _bold_axis_labels(ax, "", feat_a if feat_a else "",
                       "8. Violin Plot")

    # ---- Panel 9: Scatter plot (feature 1 vs feature 2, hued by target) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None and feat_b is not None:
        hue = df[target_col] if (target_col and is_classification_target) else None
        sns.scatterplot(x=df[feat_a], y=df[feat_b],
                         hue=hue, ax=ax, palette="Set2", legend=False, s=18)
    _bold_axis_labels(ax, feat_a if feat_a else "", feat_b if feat_b else "",
                       "9. Scatter Plot")

    # ---- Panel 10: Q-Q plot (normality check on highest-variance feature) ----
    ax = axes[panel]; panel += 1
    if feat_a is not None:
        stats.probplot(df[feat_a].dropna(), dist="norm", plot=ax)
        ax.get_lines()[0].set_markerfacecolor("#2980b9")
        ax.get_lines()[0].set_markeredgecolor("#2980b9")
        ax.get_lines()[1].set_color("#c0392b")
    _bold_axis_labels(ax, "Theoretical Quantiles", "Sample Quantiles", "10. Q-Q Plot")

    # ---- Panel 11: KDE / density plot overlay of top numeric features ----
    ax = axes[panel]; panel += 1
    for c in kde_cols:
        sns.kdeplot(df[c], ax=ax, label=c, linewidth=1.5)
    if kde_cols:
        ax.legend(prop=fm.FontProperties(family="Times New Roman", size=9))
    _bold_axis_labels(ax, "Value", "Density", "11. KDE / Density Plot")

    # ---- Panel 12: Feature importance / variance plot ----
    ax = axes[panel]; panel += 1
    if len(numeric_cols) > 0:
        var = df[numeric_cols].var().sort_values(ascending=False).head(8)
        var.plot(kind="barh", ax=ax, color="#16a085")
        ax.invert_yaxis()
    _bold_axis_labels(ax, "Variance", "Feature", "12. Variance / Importance Plot")

    for ax in axes:
        _bold_axis_labels(ax)  # re-apply tick font in case a plotting call reset it

    saved = _save_eps(fig, save_path)
    if saved:
        print(f"[generate_eda_summary] Figure saved -> {saved} (600 DPI, EPS)")
    return fig


# --------------------------------------------------------------------------
# 3. GENERIC REGRESSION TRAIN/EVAL FUNCTION  (Section 4.2)
# --------------------------------------------------------------------------
def train_evaluate_regression(models: dict, X_train, X_test, y_train, y_test,
                               scale=False, verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of regression
    models on the same train/test split.

    Parameters
    ----------
    models : dict {name: sklearn-estimator}
    X_train, X_test, y_train, y_test : array-like
    scale : bool -> StandardScaler applied when True (fit on train only)
    verbose : bool -> print per-model metrics as they are computed

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by R2 desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = regression_performance_metrics(y_test, y_pred, model_name=name,
                                                   verbose=verbose, return_dict=True)
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("R2", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 4. GENERIC CLASSIFICATION TRAIN/EVAL FUNCTION  (Section 4.3)
# --------------------------------------------------------------------------
def train_evaluate_classification(models: dict, X_train, X_test, y_train, y_test,
                                   scale=False, average="weighted", verbose=True):
    """
    Reusable function to train and evaluate an arbitrary set of classification
    models on the same train/test split.

    Returns
    -------
    results_df : pandas.DataFrame  (one row per model, sorted by Accuracy desc)
    fitted_models : dict {name: fitted estimator}
    """
    from sklearn.preprocessing import StandardScaler

    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train = scaler.transform(X_train)
        X_test = scaler.transform(X_test)

    rows, fitted_models = [], {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = None
        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)
            except Exception:
                y_proba = None
        metrics = classification_performance_metrics(
            y_test, y_pred, y_proba=y_proba, model_name=name,
            average=average, verbose=verbose, return_dict=True, plot=False
        )
        rows.append(metrics)
        fitted_models[name] = model

    results_df = pd.DataFrame(rows).set_index("Model").sort_values("Accuracy", ascending=False)
    return results_df, fitted_models


# --------------------------------------------------------------------------
# 5. GENERIC REGRESSION METRICS FUNCTION  (Section 4.4)
# --------------------------------------------------------------------------
def regression_performance_metrics(y_true, y_pred, model_name="Model",
                                    verbose=True, return_dict=False):
    """
    Computes and displays ALL standard regression performance metrics:
    MAE, MSE, RMSE, R2, Adjusted R2 (n only), MAPE.
    """
    from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                                  r2_score, mean_absolute_percentage_error)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    if verbose:
        print(f"--- Regression Metrics: {model_name} ---")
        print(f"  MAE  : {mae:.4f}")
        print(f"  MSE  : {mse:.4f}")
        print(f"  RMSE : {rmse:.4f}")
        print(f"  R2   : {r2:.4f}")
        print(f"  MAPE : {mape:.2f}%\n")

    result = {"Model": model_name, "MAE": mae, "MSE": mse,
              "RMSE": rmse, "R2": r2, "MAPE(%)": mape}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


# --------------------------------------------------------------------------
# 6. GENERIC CLASSIFICATION METRICS FUNCTION  (Section 4.5)
# --------------------------------------------------------------------------
def classification_performance_metrics(y_true, y_pred, y_proba=None,
                                        model_name="Model", average="weighted",
                                        verbose=True, return_dict=False,
                                        plot=True, save_path=None):
    """
    Computes and displays ALL standard classification performance metrics:
    Accuracy, Precision, Recall, F1-score, ROC-AUC (binary/multiclass ovr),
    and (optionally) plots the confusion matrix using the mandatory lab
    formatting (Times New Roman, bold 15pt axis labels).
    """
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score, confusion_matrix)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1 = f1_score(y_true, y_pred, average=average, zero_division=0)

    roc_auc = np.nan
    if y_proba is not None:
        try:
            n_classes = y_proba.shape[1]
            if n_classes == 2:
                roc_auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr",
                                         average=average)
        except Exception:
            roc_auc = np.nan

    if verbose:
        print(f"--- Classification Metrics: {model_name} ---")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  ROC-AUC   : {roc_auc:.4f}" if not np.isnan(roc_auc) else "  ROC-AUC   : N/A")
        print()

    if plot:
        set_plot_style()
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    annot_kws={"fontfamily": "Times New Roman", "fontsize": 13})
        _bold_axis_labels(ax, "Predicted Label", "True Label",
                           f"Confusion Matrix \u2013 {model_name}")
        _save_eps(fig, save_path)

    result = {"Model": model_name, "Accuracy": acc, "Precision": prec,
              "Recall": rec, "F1-score": f1, "ROC-AUC": roc_auc}
    return result if return_dict else pd.DataFrame([result]).set_index("Model")


In [ ]:
set_plot_style()
PALETTE = sns.color_palette("tab10", len(activities))

fig, axes = plt.subplots(2, 3, figsize=(19, 11))
fig.suptitle("Exploratory Data Analysis - UCI Human Activity Recognition",
             fontfamily="Times New Roman", fontweight="bold")

ax = axes[0, 0]
counts = df["Activity"].value_counts().reindex(activities)
ax.bar(range(len(activities)), counts.values, color=PALETTE)
ax.set_xticks(range(len(activities)))
ax.set_xticklabels([a.replace("_", "\n") for a in activities], fontsize=9)
_bold_axis_labels(ax, "Activity", "Windows", "1. Class distribution")

ax = axes[0, 1]
ax.plot(range(1, 51), pca_full.explained_variance_ratio_[:50] * 100,
        "o-", color="#4C72B0", markersize=4, label="Individual")
ax2 = ax.twinx()
ax2.plot(range(1, 51), cum_var[:50] * 100, "s--", color="#C44E52",
         markersize=4, label="Cumulative")
ax2.axhline(95, color="gray", linestyle=":")
ax2.set_ylabel("Cumulative variance (%)")
_bold_axis_labels(ax, "Principal component", "Individual variance (%)",
                  "2. PCA scree plot")

ax = axes[0, 2]
for c, a in enumerate(activities):
    m = y_true == c
    ax.scatter(X_pca2[m, 0], X_pca2[m, 1], s=4, alpha=0.4, color=PALETTE[c], label=a)
ax.legend(fontsize=8, markerscale=3)
_bold_axis_labels(ax, "PC 1", "PC 2", "3. PCA projection (true activities)")

ax = axes[1, 0]
for c, a in enumerate(activities):
    m = y_true[sub_idx] == c
    ax.scatter(X_tsne[m, 0], X_tsne[m, 1], s=5, alpha=0.6, color=PALETTE[c], label=a)
ax.legend(fontsize=8, markerscale=3)
_bold_axis_labels(ax, "t-SNE 1", "t-SNE 2", "4. t-SNE projection (true activities)")

ax = axes[1, 1]
key = "tBodyAcc-mean()-X"
key = key if key in df.columns else feature_cols[0]
for c, a in enumerate(activities):
    sns.kdeplot(df.loc[y_true == c, key], ax=ax, color=PALETTE[c], label=a, warn_singular=False)
ax.legend(fontsize=8)
_bold_axis_labels(ax, key, "Density", "5. Feature distribution by activity")

ax = axes[1, 2]
corr = np.corrcoef(X[:, :60].T)
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, cbar=True,
            xticklabels=False, yticklabels=False)
_bold_axis_labels(ax, "Feature", "Feature", "6. Correlation (first 60 features)")

fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/eda_har.eps")
plt.close(fig)

## 4. MODEL A - K-MEANS WITH THE ELBOW METHOD

In [ ]:
kmeans_rows, kmeans_models = [], {}
for k in range(2, 9):
    t0 = time.time()
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    lab = km.labels_
    kmeans_models[k] = km
    kmeans_rows.append({
        "k": k, "WCSS (Inertia)": km.inertia_,
        "Silhouette": silhouette_score(X, lab, sample_size=5000, random_state=RANDOM_STATE),
        "Davies-Bouldin": davies_bouldin_score(X, lab),
        "Calinski-Harabasz": calinski_harabasz_score(X, lab),
        "ARI": adjusted_rand_score(y_true, lab),
        "NMI": normalized_mutual_info_score(y_true, lab),
        "Time (s)": time.time() - t0,
    })
    print("  k=%d: WCSS=%.0f silhouette=%.4f ARI=%.4f NMI=%.4f (%.1fs)"
          % (k, km.inertia_, kmeans_rows[-1]["Silhouette"], kmeans_rows[-1]["ARI"],
             kmeans_rows[-1]["NMI"], kmeans_rows[-1]["Time (s)"]))

kmeans_df = pd.DataFrame(kmeans_rows)
kmeans_df.to_csv(f"{RES_DIR}/kmeans_elbow_results.csv", index=False)
print(kmeans_df.to_string(index=False))

best_k_sil = int(kmeans_df.loc[kmeans_df["Silhouette"].idxmax(), "k"])
# elbow point: largest drop in the rate of WCSS decrease (discrete 2nd difference)
wcss = kmeans_df["WCSS (Inertia)"].values
elbow_k = int(kmeans_df["k"].values[1:-1][np.argmax(np.diff(wcss, 2))])
print("Best k by silhouette: %d | elbow k (2nd difference): %d" % (best_k_sil, elbow_k))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
ax = axes[0]
ax.plot(kmeans_df["k"], kmeans_df["WCSS (Inertia)"], "o-", color="#4C72B0")
ax.axvline(elbow_k, color="#C44E52", linestyle="--", label=f"elbow at k = {elbow_k}")
ax.legend()
_bold_axis_labels(ax, "Number of clusters (k)", "WCSS (inertia)",
                  "Elbow method: k vs WCSS")
ax = axes[1]
ax.plot(kmeans_df["k"], kmeans_df["Silhouette"], "s-", color="#55A868")
ax.axvline(best_k_sil, color="#C44E52", linestyle="--", label=f"best k = {best_k_sil}")
ax.legend()
_bold_axis_labels(ax, "Number of clusters (k)", "Silhouette score",
                  "Silhouette analysis: k vs silhouette")
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/kmeans_elbow_silhouette.eps")
plt.close(fig)

In [ ]:
# Two K-Means solutions are kept: the silhouette/elbow optimum, and k = 6
# (the number of labelled activities) for a like-for-like comparison.
K_OPT = best_k_sil
labels_km_opt = kmeans_models[K_OPT].labels_
labels_km6 = kmeans_models[6].labels_
print("K-Means k=%d: ARI=%.4f NMI=%.4f" % (K_OPT, adjusted_rand_score(y_true, labels_km_opt),
                                           normalized_mutual_info_score(y_true, labels_km_opt)))

## 5. MODEL B - DBSCAN (eps and min_samples tuning)

DBSCAN is run twice: once on the full 561-dimensional standardized space
(where distance concentration is expected to hurt it) and once on the
10-component PCA space, with a grid over eps and min_samples.

In [ ]:
def dbscan_row(space, eps, min_samples, Xs):
    t0 = time.time()
    lab = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit_predict(Xs)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    noise = float(np.mean(lab == -1))
    sil = np.nan
    mask = lab != -1
    if n_clusters >= 2 and mask.sum() > 10:
        sil = silhouette_score(Xs[mask], lab[mask], sample_size=min(5000, int(mask.sum())),
                               random_state=RANDOM_STATE)
    return {"Space": space, "eps": eps, "min_samples": min_samples,
            "Clusters": n_clusters, "Noise Fraction": noise, "Silhouette": sil,
            "ARI": adjusted_rand_score(y_true, lab),
            "NMI": normalized_mutual_info_score(y_true, lab),
            "Time (s)": time.time() - t0}, lab


dbscan_rows, dbscan_labels = [], {}
for eps in [5.0, 7.5, 10.0, 12.5, 15.0]:
    row, lab = dbscan_row("561-D standardized", eps, 10, X)
    dbscan_rows.append(row)
    print("  full-D eps=%.1f -> clusters=%d noise=%.1f%% ARI=%.3f"
          % (eps, row["Clusters"], 100 * row["Noise Fraction"], row["ARI"]))

for eps in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0]:
    for ms in [5, 10, 20]:
        row, lab = dbscan_row("10-D PCA", eps, ms, X_pca10)
        dbscan_rows.append(row)
        dbscan_labels[(eps, ms)] = lab
        print("  PCA-10 eps=%.1f ms=%d -> clusters=%d noise=%.1f%% ARI=%.3f NMI=%.3f"
              % (eps, ms, row["Clusters"], 100 * row["Noise Fraction"], row["ARI"], row["NMI"]))

dbscan_df = pd.DataFrame(dbscan_rows)
dbscan_df.to_csv(f"{RES_DIR}/dbscan_tuning.csv", index=False)

valid = dbscan_df[(dbscan_df["Clusters"] >= 2) & (dbscan_df["Space"] == "10-D PCA")]
best_db = valid.loc[valid["ARI"].idxmax()]
labels_db = dbscan_labels[(best_db["eps"], int(best_db["min_samples"]))]
print("\nBest DBSCAN:", best_db.to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for ms, marker in [(5, "o"), (10, "s"), (20, "^")]:
    sub = dbscan_df[(dbscan_df["Space"] == "10-D PCA") & (dbscan_df["min_samples"] == ms)]
    ax.plot(sub["eps"], sub["ARI"], marker + "-", label=f"min_samples = {ms}")
ax.legend()
_bold_axis_labels(ax, "eps (neighbourhood radius)", "Adjusted Rand Index",
                  "DBSCAN tuning on the 10-D PCA space")
_save_eps(fig, f"{FIG_DIR}/dbscan_tuning.eps")
plt.close(fig)

## 6. MODEL C - HIERARCHICAL AGGLOMERATIVE CLUSTERING

Ward linkage needs the full pairwise distance structure, so the linkage
comparison and the dendrogram are computed on a stratified subsample of
3,000 windows; the chosen configuration is then refit on the subsample for
the metric table (the same subsample is used for every linkage so the
numbers are directly comparable).

In [ ]:
hac_idx = np.concatenate([rng.choice(np.where(y_true == c)[0], 500, replace=False)
                          for c in range(len(activities))])
rng.shuffle(hac_idx)
X_hac, y_hac = X[hac_idx], y_true[hac_idx]

hac_rows, hac_labels = [], {}
for link in ["ward", "complete", "average", "single"]:
    t0 = time.time()
    lab = AgglomerativeClustering(n_clusters=6, linkage=link).fit_predict(X_hac)
    hac_labels[link] = lab
    n_eff = len(set(lab))
    hac_rows.append({
        "Linkage": link, "Clusters": n_eff,
        "Silhouette": silhouette_score(X_hac, lab) if n_eff > 1 else np.nan,
        "Davies-Bouldin": davies_bouldin_score(X_hac, lab) if n_eff > 1 else np.nan,
        "Calinski-Harabasz": calinski_harabasz_score(X_hac, lab) if n_eff > 1 else np.nan,
        "ARI": adjusted_rand_score(y_hac, lab),
        "NMI": normalized_mutual_info_score(y_hac, lab),
        "Time (s)": time.time() - t0})
    print("  %-9s ARI=%.4f NMI=%.4f silhouette=%.4f"
          % (link, hac_rows[-1]["ARI"], hac_rows[-1]["NMI"], hac_rows[-1]["Silhouette"]))

hac_df = pd.DataFrame(hac_rows)
hac_df.to_csv(f"{RES_DIR}/hac_linkage_comparison.csv", index=False)
best_link = hac_df.loc[hac_df["ARI"].idxmax(), "Linkage"]
labels_hac = hac_labels["ward"]
print(hac_df.to_string(index=False))
print("Best linkage by ARI:", best_link)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
Z = linkage(X_hac, method="ward")
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax, color_threshold=Z[-5, 2],
           no_labels=True)
ax.axhline(Z[-5, 2], color="#C44E52", linestyle="--",
           label="cut giving 6 clusters")
ax.legend()
_bold_axis_labels(ax, "Merged clusters (last 30 merges)", "Ward linkage distance",
                  "Dendrogram - Ward linkage (3,000-window subsample)")
_save_eps(fig, f"{FIG_DIR}/dendrogram_ward.eps")
plt.close(fig)

## 7. COMPARISON OF ALL THREE ALGORITHMS

In [ ]:
def cluster_metrics(name, Xs, labels, y_ref):
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    return {"Algorithm": name, "Clusters": n_clusters,
            "Noise Fraction": float(np.mean(~mask)),
            "Silhouette": silhouette_score(Xs[mask], labels[mask], sample_size=min(5000, int(mask.sum())),
                                           random_state=RANDOM_STATE) if n_clusters > 1 else np.nan,
            "Davies-Bouldin": davies_bouldin_score(Xs[mask], labels[mask]) if n_clusters > 1 else np.nan,
            "Calinski-Harabasz": calinski_harabasz_score(Xs[mask], labels[mask]) if n_clusters > 1 else np.nan,
            "ARI": adjusted_rand_score(y_ref, labels),
            "NMI": normalized_mutual_info_score(y_ref, labels)}


comparison = pd.DataFrame([
    cluster_metrics(f"K-Means (k={K_OPT})", X, labels_km_opt, y_true),
    cluster_metrics("K-Means (k=6)", X, labels_km6, y_true),
    cluster_metrics(f"DBSCAN (eps={best_db['eps']}, min_samples={int(best_db['min_samples'])})",
                    X_pca10, labels_db, y_true),
    cluster_metrics("Hierarchical (Ward, k=6)", X_hac, labels_hac, y_hac),
])
comparison.to_csv(f"{RES_DIR}/algorithm_comparison.csv", index=False)
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
short = ["K-Means\n(k=%d)" % K_OPT, "K-Means\n(k=6)", "DBSCAN", "Hierarchical\n(Ward)"]
ax = axes[0]
xs = np.arange(len(short)); w = 0.38
ax.bar(xs - w / 2, comparison["ARI"], w, label="ARI", color="#4C72B0")
ax.bar(xs + w / 2, comparison["NMI"], w, label="NMI", color="#C44E52")
ax.set_xticks(xs); ax.set_xticklabels(short, fontsize=11)
ax.legend()
_bold_axis_labels(ax, "Algorithm", "Score vs true activity labels",
                  "External validation: ARI and NMI")
ax = axes[1]
ax.bar(xs, comparison["Silhouette"], 0.5, color="#55A868")
ax.set_xticks(xs); ax.set_xticklabels(short, fontsize=11)
_bold_axis_labels(ax, "Algorithm", "Silhouette score",
                  "Internal validation: silhouette")
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/algorithm_comparison.eps")
plt.close(fig)

In [ ]:
# Cluster assignments in 2-D (PCA and t-SNE), one panel per algorithm
fig, axes = plt.subplots(2, 3, figsize=(19, 11))
panels = [("True activities", y_true, X_pca2, activities),
          (f"K-Means (k={K_OPT})", labels_km_opt, X_pca2, None),
          ("DBSCAN (best)", labels_db, X_pca2, None)]
for ax, (title, lab, coords, names) in zip(axes[0], panels):
    for c in sorted(set(lab)):
        m = lab == c
        col = "lightgray" if c == -1 else PALETTE[c % len(PALETTE)]
        ax.scatter(coords[m, 0], coords[m, 1], s=4, alpha=0.4, color=col,
                   label=(names[c] if names is not None else
                          ("noise" if c == -1 else f"cluster {c}")))
    ax.legend(fontsize=7, markerscale=3)
    _bold_axis_labels(ax, "PC 1", "PC 2", title)

tsne_panels = [("True activities", y_true[sub_idx], activities),
               (f"K-Means (k={K_OPT})", labels_km_opt[sub_idx], None),
               ("DBSCAN (best)", labels_db[sub_idx], None)]
for ax, (title, lab, names) in zip(axes[1], tsne_panels):
    for c in sorted(set(lab)):
        m = lab == c
        col = "lightgray" if c == -1 else PALETTE[c % len(PALETTE)]
        ax.scatter(X_tsne[m, 0], X_tsne[m, 1], s=5, alpha=0.6, color=col,
                   label=(names[c] if names is not None else
                          ("noise" if c == -1 else f"cluster {c}")))
    ax.legend(fontsize=7, markerscale=3)
    _bold_axis_labels(ax, "t-SNE 1", "t-SNE 2", title + " (t-SNE)")
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/cluster_visualisation.eps")
plt.close(fig)

In [ ]:
# Cluster-to-activity contingency tables
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (title, lab, y_ref) in zip(axes, [
        (f"K-Means (k={K_OPT})", labels_km_opt, y_true),
        ("DBSCAN (best)", labels_db, y_true),
        ("Hierarchical (Ward, k=6)", labels_hac, y_hac)]):
    ct = pd.crosstab(pd.Series([activities[i] for i in y_ref], name="Activity"),
                     pd.Series(lab, name="Cluster"))
    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                annot_kws={"fontfamily": "Times New Roman", "fontsize": 9})
    ax.tick_params(labelsize=9)
    _bold_axis_labels(ax, "Cluster", "True activity", title)
fig.tight_layout()
_save_eps(fig, f"{FIG_DIR}/cluster_activity_contingency.eps")
plt.close(fig)

## 8. SUMMARY OF NUMBERS USED IN THE REPORT

In [ ]:
ct_km = pd.crosstab(pd.Series([activities[i] for i in y_true]), pd.Series(labels_km_opt))
ct_km.to_csv(f"{RES_DIR}/kmeans_contingency.csv")
ct_km6 = pd.crosstab(pd.Series([activities[i] for i in y_true]), pd.Series(labels_km6))
ct_km6.to_csv(f"{RES_DIR}/kmeans6_contingency.csv")
print(ct_km6.to_string())

summary = {
    "n_samples": int(X.shape[0]), "n_features": int(X.shape[1]),
    "n_activities": int(len(activities)), "activities": list(map(str, activities)),
    "class_counts": {str(k): int(v) for k, v in df["Activity"].value_counts().items()},
    "pca_components_95": n_95,
    "pca_var_pc1": float(cum_var[0]), "pca_var_pc2": float(cum_var[1]),
    "kmeans_table": kmeans_df.to_dict("records"),
    "elbow_k": elbow_k, "best_k_silhouette": best_k_sil,
    "dbscan_best": {k: (float(v) if isinstance(v, (int, float, np.floating)) else str(v))
                    for k, v in best_db.to_dict().items()},
    "dbscan_full_dim": dbscan_df[dbscan_df["Space"] == "561-D standardized"].to_dict("records"),
    "hac_table": hac_df.to_dict("records"),
    "comparison": comparison.to_dict("records"),
    "kmeans_contingency": ct_km.to_dict(),
    "kmeans6_contingency": ct_km6.to_dict(),
    "static_vs_dynamic": {
        "note": "cluster purity of the 2-cluster K-Means solution",
        "ari_k2": float(kmeans_df.loc[kmeans_df["k"] == 2, "ARI"].iloc[0]),
        "nmi_k2": float(kmeans_df.loc[kmeans_df["k"] == 2, "NMI"].iloc[0])},
}
with open(f"{RES_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str)[:4000])